# Pipeline Schubert — revisión del manuscrito

Ejecuta las celdas en orden. Todo lo costoso (muestreo, FVA) queda en `cache/` y se reutiliza si vuelves a correr.

**Archivos necesarios en esta carpeta:** `pipeline_schubert.py`, `paso1_codim_normalizada.py`, `figuras_revision.py` y tu modelo `community12.xml`.

**Instalación (una vez):** `pip install cobra highspy statsmodels ripser persim adjustText`
Si tienes licencia académica de Gurobi (gratuita), instala `gurobipy` y usa `SOLVER = "gurobi"`: es mucho más rápido.

In [ ]:
# 1. Configuración
RUTA_MODELO = "community12.xml"
SOLVER      = "hybrid"     # "hybrid" = HiGHS (gratis); "gurobi" si tienes licencia; "glpk" es muy lento
INTERFAZ    = "todas"      # "todas" (recomendado) | "tex" (reproduce la Tabla 5 enviada)
N           = 150          # estados por escenario (~7 s c/u con HiGHS)
EPS         = 1e-6
CACHE       = "cache"
B_PERM      = 1000         # réplicas en la sensibilidad de la bandera
CORRER_CONTROL = True      # control de perturbación aleatoria (~12 min por réplica con HiGHS)
B_CTRL, N_CTRL = 10, 30
CORRER_FVA  = False        # envolvente FVA (costoso: 2 optimizaciones por reacción de interfaz)

import os, numpy as np, pandas as pd
import pipeline_schubert as ps
import paso1_codim_normalizada as p1
import figuras_revision as fr
os.makedirs("resultados", exist_ok=True)

In [ ]:
# 2. Reparar el SBML, cargar el modelo, aplicar medio M9 + glicerol
ruta_fija = RUTA_MODELO.replace(".xml", "_fixed.xml")
if not os.path.exists(ruta_fija):
    ps.reparar_sbml(RUTA_MODELO, ruta_fija)
model = ps.cargar_modelo(ruta_fija, solver=SOLVER)
ps.aplicar_medio(model)              # edita ps.M9_GLICEROL si usaste otros límites
print("crecimiento comunitario óptimo:", model.slim_optimize())

In [ ]:
# 3. Espacio de intercambio E y bandera de referencia
mapa = ps.construir_mapa(model, interfaz=INTERFAZ)
display(ps.resumen_mapa(mapa))
print("niveles de la bandera (sin leave-one-out):", ps.dimensiones_bandera(mapa))
import pickle
pickle.dump(mapa, open("mapa_%s.pkl" % INTERFAZ, "wb"))

In [ ]:
# 4. Muestreo random-FBA por lotes de 40 estados en procesos nuevos.
#    (la interfaz optlang/HiGHS acumula ~25 MB por estado; por lotes la memoria no crece)
#    Se reanuda desde cache/ si se interrumpe.
import subprocess, sys
for esc in ["SS19", "SS37"]:
    subprocess.run([sys.executable, "muestrear_lote.py", esc, RUTA_MODELO.replace(".xml", "_fixed.xml"),
                    INTERFAZ, str(N)], check=True)
F = {esc: np.load(f"{CACHE}/flujos_{esc}_n{N}.npz")["F"] for esc in ["SS19", "SS37"]}

In [ ]:
# 5. muestras.csv: una fila por (estado, especie) con d, |lambda|, nu, z
muestras = ps.tabla_muestras(mapa, F, eps=EPS, guardar="resultados/muestras.csv")
muestras.groupby(["especie", "escenario"])[["d", "codim", "nu"]].mean().round(3)

### Tabla de tolerancia a NaCl
Valores (% p/v) del intervalo de crecimiento de la cepa tipo, o de una cepa tipo congenérica cuando no hay dato a nivel de especie. *Natronomonas* (sin límite superior) y *Roseovarius* (sin valor comparable) quedan excluidas. Fuentes en la Tabla 3 del manuscrito.

In [ ]:
# 6. Tolerancia
tol = pd.DataFrame([
    ("Hqr. walsbyi", 14, 32, 1), ("Sal. ruber", 20, 30, 1), ("Hrr. salinarum", 10, 30, 1),
    ("Spb. salinus", 10, 25, 1), ("Hmc. mukohataei", 8.2, 19.9, 0),
    ("Arh. aquaeolei", 6, 20, 1), ("Pcc. vermicola", 1.0, 7.5, 1),
    ("Ocn. granulosus", 0.5, 10, 0), ("Gra. halophila", 1, 20, 0),
], columns=["especie", "min", "max", "alta_confianza"])
tol["tolerancia"] = tol["max"] - tol["min"]
tol.to_csv("tolerancia.csv", index=False)
tol

In [ ]:
# 7. Paso 1 — H1a con codimensión normalizada (cruda / d / nu / z / nu | d)
esp, corr = p1.analizar(muestras, tol, mapa.m, "resultados/paso1")
display(corr.round(4))

In [ ]:
# 8. Sensibilidad de la bandera (punto 1 del revisor)
res_bandera, det_bandera = ps.sensibilidad_bandera(mapa, F, tol, B=B_PERM, eps=EPS,
                                                    guardar="resultados/sensibilidad_bandera.csv")
res_bandera.round(3)

In [ ]:
# 9. Sensibilidad al umbral epsilon
eps_tab = ps.barrido_epsilon(mapa, F)
eps_tab.to_csv("resultados/barrido_epsilon.csv", index=False)
eps_tab.groupby("eps")[["d_media", "nu_media", "cambio_nu"]].agg(["mean", "max"]).round(4)

In [ ]:
# 10. Efecto de escenario (regresión ordinal) y control de perturbación aleatoria
ordinal = ps.regresion_ordinal(muestras)
if CORRER_CONTROL:
    import glob
    for rep in range(B_CTRL):     # un proceso nuevo por réplica (memoria)
        subprocess.run([sys.executable, "control_run.py", str(rep), str(N_CTRL)], check=True)
    ctrl = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("resultados/control_rep*.csv"))])
    ordinal = ps.comparar_con_control(ordinal, ctrl)
ordinal.to_csv("resultados/regresion_ordinal.csv", index=False)
ordinal.round(3)

In [ ]:
# 11. Envolvente FVA (punto 3 del revisor)
if CORRER_FVA:
    fva = pd.concat([ps.envolvente_fva(model, mapa, esc, eps=EPS, cache_dir=CACHE)
                     for esc in ["SS19", "SS37"]])
    fva.to_csv("resultados/envolvente_fva.csv", index=False)
    comp = (muestras.groupby(["especie", "escenario"])["nu"].mean().rename("nu_randomFBA")
            .reset_index().merge(fva[["especie", "escenario", "d", "nu"]]
                                 .rename(columns={"d": "d_FVA", "nu": "nu_FVA"})))
    display(comp.round(3))

In [ ]:
# 12. Homología persistente y bottleneck.csv
F_todos = np.concatenate([F["SS19"], F["SS37"]])
ph, D = ps.homologia_persistente(mapa, F_todos, guardar_bottleneck="resultados/bottleneck.csv")
ph = ph.merge(muestras.groupby("especie")[["d", "nu"]].mean().reset_index(), on="especie")
ph.to_csv("resultados/homologia_persistente.csv", index=False)
ph.round(3)

In [ ]:
# 13. Figuras del manuscrito (PDF + PNG en figs/)
fr.style()
fr.fig_schema("figs")
fr.fig_diagnostic("figs")
dfs = fr.load_samples("resultados/muestras.csv", None, mapa.m)
fr.fig_nu_species(dfs, "figs")
fr.fig_h1a(dfs, tol, mapa.m, "figs")
fr.fig_bottleneck(pd.read_csv("resultados/bottleneck.csv", index_col=0), "figs")
print("listo:", sorted(os.listdir("figs")))